# 02 REBUILD: fault log, episodes and labels

Builds the fault log used in the paper from the raw export `37011FL.csv`, pairs each onset with its recovery into an episode, and labels the June 2025 telemetry for the four target windows.

* **Records.** The logging period starts on 4 November 2024; records logged in the maintenance shed are removed.
* **Exclusion rule.** Operating events and commands, which do not report an abnormal condition, are removed: Power On, MCE Power Off, MCE Off - Pan Down 10 Min, Shunting Mode is active, PTDC Mode is Active, Alarm Chain Pulling, and every "Cut In / Cut Out from Display" command. Every other message the vehicle control unit logs is kept, including protective states caused by the crew (emergency stop, vigilance, start/run interlocks).
* **Episodes.** Each onset is paired with the next recovery of the same fault, no later than the next onset of that fault and within a six-hour cap. An onset without a recovery is kept with zero duration.
* **Labels.** D5 (five seconds either side of onset, a detection reference) and the warning windows W5, W60 and W300, each ending one second before onset. Seconds inside an episode are excluded from the warning windows; D5 keeps the five seconds after onset.

Locations for records present in an earlier version of the log (`DATA/37011FLv3.csv`, an input) are taken from it, since they were filled by hand there. The differences between that version and the log used in the paper are listed record by record.

Outputs used in the paper: `DATA/37011FLv4.csv`, `build/fault_episodes_v4.csv`, `build/telemetry_labelled_v4.pkl`.

Paper: Sections 2.5 and 2.7.1.

In [ ]:
import os, sys, re
import numpy as np, pandas as pd, sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

env = os.environ.get
A1, A2 = "../ANALYSIS1", "../ANALYSIS2"
DATA, BUILD = env("REV_DATADIR", "DATA"), env("REV_BUILDDIR", "build")
CFG = dict(
    raw_log   = f"{A1}/DATA/37011/37011FL.csv",
    flv3      = f"{DATA}/37011FLv3.csv",
    episodes_v3 = f"{BUILD}/fault_episodes_v3.csv",
    telemetry = f"{A2}/TELEMETRY/build/telemetry_raw_v2.pkl",
    outdir    = env("REV_OUTDIR", "OUTPUT") + "/v4",
    shed      = "ELECTRIC LOCO SHED TATA NAGAR",
    start     = "2024-11-04 05:09:06",          # first record of the logging period (37011FLv1)
    june      = ("2025-06-01 00:00:00", "2025-06-30 23:59:59"),
    max_episode_s = 6 * 3600, debounce_s = 60,
    blower    = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons  = {"D5": (5, 5, "detection"), "W5": (5, -1, "warning"),
                 "W60": (60, -1, "warning"), "W300": (300, -1, "warning")},
)
EXCLUDED = ["POWER ON", "MCE POWER OFF", "MCE OFF - PAN DOWN 10 MIN", "SHUNTING MODE IS ACTIVE",
            "PTDC MODE IS ACTIVE", "ALARM CHAIN PULLING"]
COMMAND = re.compile(r"CUT (?:IN|OUT) FROM DISPLAY$")
os.makedirs(CFG["outdir"], exist_ok=True)
for d in (DATA, BUILD):
    os.makedirs(d, exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

def base(s):
    return s.str.replace(r"\s*recovered\s*$", "", flags=re.I, regex=True).str.strip().str.upper()
def excluded(b):
    return b.isin(EXCLUDED) | b.str.contains(COMMAND)

## 1. Build the log from the raw export

In [ ]:
raw = pd.read_csv(CFG["raw_log"])
raw1 = raw[raw["LOCATION"] != CFG["shed"]].reset_index(drop=True)
t0 = raw1["DESCRIPTION"].str.split(":")
raw1 = raw1.assign(FAULT=t0.apply(lambda x: x[1]), T1=t0.apply(lambda x: x[0]))
raw1["SYSTEMPART"] = raw1["T1"].str.lstrip("[FP0123456789NODIS] ")
raw1 = raw1.drop(columns=["DESCRIPTION", "T1"])
X = TfidfVectorizer(stop_words="english").fit_transform(raw1["FAULT"])
raw1["CLUSTER"] = KMeans(n_clusters=20, random_state=42).fit_predict(X)     # grouping key only
raw1["dt"] = pd.to_datetime(raw1["DATETIME"], format="%m/%d/%Y %H:%M:%S")
raw1["BASE"] = base(raw1["FAULT"])

per = raw1[raw1.dt >= pd.Timestamp(CFG["start"])].copy()
ex = excluded(per.BASE)
rule = (per[ex].assign(rec=per.FAULT.str.contains("Recovered", case=False))
          .groupby("BASE").agg(records=("FAULT", "size"), onsets=("rec", lambda s: int((~s).sum())))
          .sort_values("onsets", ascending=False))
print(f"Logging period: {len(per):,} records; excluded by the rule: {int(ex.sum()):,} "
      f"({int(rule.onsets.sum()):,} onsets)\n"); print(rule.to_string())
rule.to_csv(f"{CFG['outdir']}/rev_rebuild_v4_excluded.csv")
v4 = per[~ex].copy()

## 2. Locations

Records also present in the earlier version of the log take its (hand-filled) location. Any other record keeps its raw location or, if that is missing, takes the location of the nearest earlier-version record in time.

In [ ]:
v3 = pd.read_csv(CFG["flv3"])
v3["dt"] = pd.to_datetime(v3["DATETIME"], format="%m/%d/%Y %H:%M:%S")
key = ["DATETIME", "CODE", "FAULT"]
v3k = v3.assign(n=v3.groupby(key).cumcount()); v4k = v4.assign(n=v4.groupby(key).cumcount())
m = v4k.merge(v3k[key + ["n", "LOCATION"]].rename(columns={"LOCATION": "LOC3"}), on=key + ["n"], how="left", indicator=True)
in_v3 = m["_merge"].eq("both").values
v4["LOCATION"] = np.where(in_v3, m["LOC3"].values, v4["LOCATION"].values)
need = v4["LOCATION"].isna()
if need.any():
    ref = v3[["dt", "LOCATION"]].sort_values("dt")
    near = pd.merge_asof(v4.loc[need, ["dt"]].sort_values("dt"), ref, on="dt", direction="nearest")
    v4.loc[v4.loc[need].sort_values("dt").index, "LOCATION"] = near["LOCATION"].values
print(f"log records: {len(v4):,}; also in the earlier version: {int(in_v3.sum()):,}; new: {int((~in_v3).sum()):,}; "
      f"new with a location filled from the nearest v3 record: {int(need.sum())}")

# every difference from the earlier version must follow from the rule
gone = v3k.merge(v4k[key + ["n"]], on=key + ["n"], how="left", indicator=True)
gone = gone[gone["_merge"].eq("left_only")]
new = v4[~in_v3]
assert excluded(base(gone.FAULT)).all(), "a v3 record was dropped that the rule does not exclude"
assert not excluded(new.BASE).any()
assert set(new.CLUSTER) <= {1, 4, 8, 16, 18}, "a new record did not come from a discarded cluster"
print(f"earlier-version records dropped by the rule: {len(gone)}; records restored from discarded clusters: {len(new)}")
diff = pd.concat([gone.assign(change="dropped", BASE=base(gone.FAULT)).groupby(["change", "BASE"]).size(), new.assign(change="restored").groupby(["change", "BASE"]).size()]).rename("records")
print(diff.to_string())
diff.to_csv(f"{CFG['outdir']}/rev_rebuild_v4_changes.csv")
same_prev = (v4[key].astype(str) == v4[key].astype(str).shift()).all(axis=1)
print(f"records identical to the record above: {int(same_prev.sum())} (raw export: 0)")

flv4 = v4[["DATETIME", "CODE", "LOCATION", "FAULT", "SYSTEMPART", "CLUSTER"]].reset_index(drop=True)
assert flv4.LOCATION.notna().all()
flv4.to_csv(f"{DATA}/37011FLv4.csv", index=False)

## 3. Episodes and June labels

In [ ]:
def build_episodes(fl, cfg):
    fl = fl.copy()
    fl["dt"] = pd.to_datetime(fl["DATETIME"], format="%m/%d/%Y %H:%M:%S")
    fl["is_rec"] = fl["FAULT"].str.contains("Recovered", case=False, na=False)
    fl["base"] = (fl["FAULT"].str.replace(" Recovered", "", regex=False).str.strip().str.upper())
    fl = fl.sort_values("dt").reset_index(drop=True)
    episodes = []
    for (cluster, b), grp in fl.groupby(["CLUSTER", "base"]):
        onsets = grp.loc[~grp.is_rec, "dt"].tolist(); recovs = grp.loc[grp.is_rec, "dt"].tolist()
        ri = 0
        for k, on in enumerate(onsets):
            nxt = onsets[k + 1] if k + 1 < len(onsets) else pd.Timestamp.max
            while ri < len(recovs) and recovs[ri] < on:
                ri += 1
            end, dur, matched = on, 0.0, False
            if ri < len(recovs):
                cand = recovs[ri]; d = (cand - on).total_seconds()
                if d <= cfg["max_episode_s"] and cand <= nxt:
                    end, dur, matched = cand, d, True; ri += 1
            episodes.append(dict(cluster=cluster, fault=b, onset=on, recovery=end, duration_s=dur, matched=matched))
    ep = pd.DataFrame(episodes).sort_values("onset").reset_index(drop=True)
    ep["debounce_keep"] = True
    for c, g in ep.groupby("cluster"):
        gaps = g["onset"].diff().dt.total_seconds()
        ep.loc[g.index, "debounce_keep"] = gaps.isna() | (gaps > cfg["debounce_s"])
    return ep

cols = ["cluster", "fault", "onset", "recovery", "duration_s", "matched", "debounce_keep"]
srt = lambda e: e[cols].sort_values(["onset", "cluster", "fault"]).reset_index(drop=True).astype(str)
ep3 = pd.read_csv(CFG["episodes_v3"], parse_dates=["onset", "recovery"])
assert srt(build_episodes(v3, CFG)).equals(srt(ep3)), "pairing does not reproduce fault_episodes_v3"
ep4 = build_episodes(flv4, CFG)
ep4.to_csv(f"{BUILD}/fault_episodes_v4.csv", index=False)

j0, j1 = (pd.Timestamp(x) for x in CFG["june"])
def summary(ep, name):
    jun = ep.onset.between(j0, j1); b = ep.fault.eq(CFG["blower"])
    return dict(log=name, episodes=len(ep), labels=ep.fault.nunique(), unmatched=int((~ep.matched).sum()),
                blower_pct=round(100 * b.mean(), 1), june=int(jun.sum()),
                june_unmatched=int((~ep[jun].matched).sum()), june_blower_pct=round(100 * b[jun].mean(), 1))
cmp = pd.DataFrame([summary(ep3, "v3"), summary(ep4, "v4")]).set_index("log")
print(cmp.T.to_string()); cmp.to_csv(f"{CFG['outdir']}/rev_rebuild_v4_episodes.csv")

In [ ]:
ONE_S = np.timedelta64(1, "s")
def label(index, ep, pre_s, post_s, kind):
    n = len(index); win = np.zeros(n, bool); epi = np.zeros(n, bool); lo = index.searchsorted
    for on, rec in zip(ep["onset"].values, ep["recovery"].values):
        win[lo(on - pre_s * ONE_S, "left"):lo(on + post_s * ONE_S, "right")] = True
        epi[lo(on, "left"):lo(max(rec, on), "right")] = True
    y = np.zeros(n, dtype=np.int8)
    if kind == "detection":
        y[epi] = -1; y[win] = 1
    else:
        y[win] = 1; y[epi] = -1
    return y

tel = pd.read_pickle(CFG["telemetry"])
tel = tel[(tel.DATETIME >= CFG["june"][0]) & (tel.DATETIME <= CFG["june"][1])].sort_values("DATETIME").reset_index(drop=True)
idx = tel["DATETIME"].values
span = ep4[ep4.onset.between(j0, j1)]
old = pd.read_pickle(f"{BUILD}/telemetry_labelled_v3.pkl").sort_values("DATETIME").reset_index(drop=True)
assert np.array_equal(old["DATETIME"].values, idx)
rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    tel[h] = label(idx, span, pre, post, kind)
    rows.append(dict(label=h, positive_v3=int((old[h] == 1).sum()), positive_v4=int((tel[h] == 1).sum()),
                     excluded_v3=int((old[h] == -1).sum()), excluded_v4=int((tel[h] == -1).sum()),
                     rows_changed=int((old[h].values != tel[h].values).sum())))
del old
lab = pd.DataFrame(rows).set_index("label"); print(lab.to_string())
lab.to_csv(f"{CFG['outdir']}/rev_rebuild_v4_labels.csv")
tel.to_pickle(f"{BUILD}/telemetry_labelled_v4.pkl")
print("written:", f"{DATA}/37011FLv4.csv", f"{BUILD}/fault_episodes_v4.csv", f"{BUILD}/telemetry_labelled_v4.pkl")

___
**END**